In [3]:
# ================================================================
# CELL 1: Install libraries — run once, takes about 30 seconds
# ================================================================
!pip install flask flask-cors pyngrok --quiet
print("✓ flask      — web server")
print("✓ flask-cors — cross-origin requests")
print("✓ pyngrok    — public URL")
print("\n✅ Done! Run Cell 2 now.")

✓ flask      — web server
✓ flask-cors — cross-origin requests
✓ pyngrok    — public URL

✅ Done! Run Cell 2 now.


In [5]:
import os
os.makedirs('templates', exist_ok=True)

html_content = r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>SENTINEL ICU — AI Vital Stack System</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&display=swap" rel="stylesheet">
    <style>
        *, *::before, *::after { margin:0; padding:0; box-sizing:border-box; }
        body { font-family:'Inter',sans-serif; background:#EEF4FF; color:#0F172A; min-height:100vh; }

        .hdr {
            background:linear-gradient(135deg,#003566 0%,#0077B6 60%,#00B4D8 100%);
            color:white; padding:0 32px;
            display:flex; justify-content:space-between; align-items:center;
            min-height:70px; box-shadow:0 4px 24px rgba(0,53,102,0.45);
        }
        .hdr-logo { font-size:10px; font-weight:800; letter-spacing:3px; opacity:0.65; }
        .hdr-title{ font-size:20px; font-weight:900; letter-spacing:-0.5px; }
        .hdr-sub  { font-size:11px; opacity:0.75; margin-top:2px; }
        .hdr-right{ display:flex; gap:12px; align-items:center; }
        .live-pill {
            display:flex; align-items:center; gap:7px;
            background:rgba(255,255,255,0.15); padding:6px 14px;
            border-radius:20px; font-size:12px; font-weight:700;
        }
        .live-dot { width:8px; height:8px; background:#06D6A0; border-radius:50%; animation:blink 1s infinite; }
        @keyframes blink { 0%,100%{opacity:1} 50%{opacity:0.2} }
        .chip { padding:7px 16px; border-radius:20px; font-size:11px; font-weight:800; white-space:nowrap; }
        .chip-ok   { background:#06D6A0; color:#003022; }
        .chip-warn { background:#FFB703; color:#3D2700; }
        .chip-crit { background:#EF233C; color:white; animation:flashchip 0.7s infinite; }
        @keyframes flashchip { 0%,100%{opacity:1} 50%{opacity:0.5} }

        .nav-bar {
            background:white; border-bottom:3px solid #DBEAFE;
            padding:0 32px; display:flex; align-items:center; gap:2px;
            box-shadow:0 2px 8px rgba(0,0,0,0.05);
        }
        .ntab {
            padding:14px 24px; font-size:13px; font-weight:700;
            cursor:pointer; background:none; border:none; color:#64748B;
            border-bottom:3px solid transparent; margin-bottom:-3px;
            font-family:'Inter',sans-serif; transition:all 0.2s;
        }
        .ntab:hover { color:#0077B6; background:#F0F7FF; }
        .ntab.active { color:#0077B6; border-bottom-color:#0077B6; background:#F0F7FF; }
        .nbadge { display:inline-block; background:#EF233C; color:white; font-size:10px; font-weight:800; padding:1px 6px; border-radius:10px; margin-left:6px; animation:flashchip 0.7s infinite; }

        .ibar { background:white; padding:9px 32px; display:flex; flex-wrap:wrap; gap:22px; border-bottom:2px solid #EEF4FF; font-size:12px; color:#475569; }
        .iv      { font-weight:700; color:#0077B6; }
        .iv-crit { font-weight:700; color:#EF233C; }
        .iv-warn { font-weight:700; color:#FFB703; }
        .iv-ok   { font-weight:700; color:#06D6A0; }
        .iv-vac  { font-weight:700; color:#94A3B8; }

        #alertBox { display:none; padding:12px 32px; font-size:13px; font-weight:700; align-items:center; gap:12px; }
        #alertBox.aw { display:flex; background:#FFF7E0; border-left:5px solid #FFB703; color:#78350F; }
        #alertBox.ac { display:flex; background:#FFF0F0; border-left:5px solid #EF233C; color:#9B1C1C; animation:flashbg 0.8s infinite; }
        @keyframes flashbg { 0%,100%{opacity:1} 50%{opacity:0.65} }

        .page { display:none; }
        .page.active { display:block; }
        .main { padding:22px 32px; max-width:1600px; margin:0 auto; }

        /* ── OVERVIEW STATS ── */
        .ward-stats { display:grid; grid-template-columns:repeat(5,1fr); gap:13px; margin-bottom:16px; }
        .ws { background:white; border-radius:14px; padding:14px 16px; box-shadow:0 2px 12px rgba(0,0,0,0.06); border:2px solid #E2EAF8; text-align:center; }
        .ws-num { font-size:32px; font-weight:900; line-height:1.1; }
        .ws-lbl { font-size:10px; font-weight:700; color:#64748B; text-transform:uppercase; letter-spacing:0.8px; margin-top:4px; }
        .ws-total .ws-num { color:#0077B6; }
        .ws-crit  .ws-num { color:#EF233C; }
        .ws-warn  .ws-num { color:#FFB703; }
        .ws-stab  .ws-num { color:#06D6A0; }
        .ws-vac   .ws-num { color:#94A3B8; }

        .ov-controls { display:flex; gap:12px; align-items:center; margin-bottom:16px; flex-wrap:wrap; }
        .oc-lbl { font-size:13px; font-weight:700; color:#334155; }

        /* ── BED CARDS (50 beds, 4-col) ── */
        .beds-grid { display:grid; grid-template-columns:repeat(4,1fr); gap:12px; }
        @media(max-width:1400px){ .beds-grid{grid-template-columns:repeat(3,1fr);} }
        @media(max-width:1000px){ .beds-grid{grid-template-columns:repeat(2,1fr);} }
        @media(max-width:640px) { .beds-grid{grid-template-columns:1fr;} }

        .bed-card {
            background:white; border-radius:12px; padding:12px 12px 12px 16px;
            box-shadow:0 2px 12px rgba(0,0,0,0.07); border:2px solid #E2EAF8;
            transition:transform 0.2s,box-shadow 0.2s; position:relative; overflow:hidden;
        }
        .bed-card::before { content:''; position:absolute; top:0;left:0;bottom:0; width:4px; background:var(--bc-ac,#E2EAF8); }
        .bed-card:hover { transform:translateY(-2px); box-shadow:0 6px 20px rgba(0,0,0,0.12); }
        .bed-card.critical { --bc-ac:#EF233C; border-color:rgba(239,35,60,0.25); }
        .bed-card.warning  { --bc-ac:#FFB703; border-color:rgba(255,183,3,0.3); }
        .bed-card.stable   { --bc-ac:#06D6A0; }
        .bed-card.vacant   { --bc-ac:#94A3B8; opacity:0.65; }
        .bed-card.nodata   { --bc-ac:#94A3B8; }

        .bc-hdr  { display:flex; justify-content:space-between; align-items:center; margin-bottom:5px; padding-left:5px; }
        .bc-bed  { font-size:10px; font-weight:900; color:#64748B; letter-spacing:1.5px; }
        .bc-pill { font-size:10px; font-weight:800; padding:2px 8px; border-radius:20px; }
        .bc-pill.critical { background:#FEE2E2; color:#9B1C1C; }
        .bc-pill.warning  { background:#FEF3C7; color:#92400E; }
        .bc-pill.stable   { background:#D1FAE5; color:#065F46; }
        .bc-pill.vacant   { background:#F1F5F9; color:#64748B; }
        .bc-pill.nodata   { background:#F1F5F9; color:#64748B; }
        .bc-name   { font-size:13px; font-weight:800; color:#0F172A; margin-bottom:2px; padding-left:5px; }
        .bc-diag   { font-size:10px; color:#64748B; margin-bottom:8px; padding-left:5px; }
        .bc-vitals { display:flex; gap:11px; margin-bottom:8px; padding-left:5px; }
        .bv { display:flex; flex-direction:column; align-items:center; }
        .bv-icon { font-size:11px; }
        .bv-val  { font-size:13px; font-weight:800; color:#0F172A; line-height:1.2; }
        .bv-val.dng { color:#EF233C; }
        .bv-unit { font-size:9px; color:#94A3B8; text-transform:uppercase; }
        .bc-time { font-size:10px; color:#94A3B8; padding-left:5px; margin-bottom:7px; }

        /* ── PATIENT SELECTOR ── */
        .pat-sel-bar { background:white; border-radius:12px; padding:11px 16px; border:2px solid #E2EAF8; margin-bottom:15px; display:flex; align-items:center; gap:12px; flex-wrap:wrap; box-shadow:0 2px 10px rgba(0,0,0,0.05); }
        .psl  { font-size:11px; font-weight:800; color:#64748B; text-transform:uppercase; letter-spacing:0.5px; }
        .psel { padding:8px 13px; border:2px solid #DBEAFE; border-radius:8px; font-size:13px; font-weight:700; color:#0F172A; font-family:'Inter',sans-serif; outline:none; background:white; cursor:pointer; min-width:220px; }
        .psel:focus { border-color:#0077B6; }
        .psi { display:flex; gap:14px; font-size:11px; color:#475569; flex-wrap:wrap; }
        .psi .pv { font-weight:700; color:#0077B6; }

        /* ── VITAL CARDS ── */
        .vcards { display:grid; grid-template-columns:repeat(3,1fr); gap:13px; margin-bottom:17px; }
        @media(max-width:1100px){ .vcards{grid-template-columns:repeat(2,1fr);} }
        .vc { background:white; border-radius:14px; padding:17px 19px; box-shadow:0 2px 14px rgba(0,0,0,0.07); border:2px solid #E2EAF8; position:relative; overflow:hidden; transition:transform 0.2s,box-shadow 0.2s; }
        .vc:hover { transform:translateY(-2px); box-shadow:0 6px 22px rgba(0,0,0,0.11); }
        .vc::after { content:''; position:absolute; bottom:0;left:0;right:0; height:4px; background:var(--ac,#0077B6); }
        .vc.ok   { --ac:#06D6A0; }
        .vc.warn { --ac:#FFB703; border-color:rgba(255,183,3,0.3); background:#FFFDF5; }
        .vc.crit { --ac:#EF233C; border-color:rgba(239,35,60,0.25); background:#FFF5F7; }
        .vc-icon { font-size:24px; margin-bottom:6px; }
        .vc-lbl  { font-size:10px; font-weight:800; text-transform:uppercase; letter-spacing:1.2px; color:#94A3B8; margin-bottom:3px; }
        .vc-val  { font-size:36px; font-weight:900; color:#0F172A; line-height:1.05; }
        .vc-val.md { font-size:23px; }
        .vc-unit { font-size:11px; color:#94A3B8; margin-top:1px; }
        .vc-note { font-size:11px; font-weight:700; margin-top:8px; }
        .vc.ok   .vc-note { color:#065F46; }
        .vc.warn .vc-note { color:#92400E; }
        .vc.crit .vc-note { color:#9B1C1C; }
        @keyframes hb { 0%,100%{transform:scale(1)} 50%{transform:scale(1.3)} }
        .hb { display:inline-block; animation:hb 1s infinite; }

        /* ── TWO COLUMN ── */
        .twoCol { display:grid; grid-template-columns:1.15fr 1fr; gap:17px; margin-bottom:17px; }
        @media(max-width:1060px){ .twoCol{grid-template-columns:1fr;} }

        .sc { background:white; border-radius:14px; padding:19px; box-shadow:0 2px 14px rgba(0,0,0,0.07); border:2px solid #E2EAF8; }
        .sc-ttl { font-size:13px; font-weight:800; color:#0077B6; margin-bottom:13px; padding-bottom:9px; border-bottom:2px solid #EEF4FF; display:flex; align-items:center; gap:7px; }

        /* ── STACK VIZ ── */
        .sthdr { display:flex; justify-content:space-between; align-items:center; margin-bottom:8px; }
        .top-badge { background:#0077B6; color:white; font-size:10px; font-weight:800; padding:3px 10px; border-radius:5px; }
        .stlist { display:flex; flex-direction:column; gap:6px; max-height:300px; overflow-y:auto; padding-right:2px; }
        .si { border:2px solid #E2EAF8; border-radius:9px; padding:8px 11px; display:flex; justify-content:space-between; align-items:center; background:#F8FAFF; animation:pushAnim 0.35s ease; }
        .si.top { border-color:#0077B6; background:#EBF5FF; box-shadow:0 3px 12px rgba(0,119,182,0.15); }
        @keyframes pushAnim { from{opacity:0;transform:translateY(-12px)} to{opacity:1;transform:translateY(0)} }
        .si-vitals { display:flex; gap:13px; }
        .sv-lbl { font-size:9px; text-transform:uppercase; color:#94A3B8; font-weight:700; }
        .sv-val { font-size:12px; font-weight:800; color:#0F172A; }
        .sv-val.r { color:#EF233C; }
        .si-badge { font-size:10px; font-weight:800; padding:2px 7px; border-radius:4px; }
        .b-top { background:#0077B6; color:white; }
        .b-num { background:#EEF4FF; color:#64748B; }
        .st-bot { text-align:center; font-size:10px; color:#94A3B8; margin-top:8px; }

        /* ── POP INDICATOR ── */
        .pop-indicator {
            background:#FFF7E0; border:2px solid #FFB703; border-radius:8px;
            padding:8px 12px; margin-top:10px; font-size:11px; font-weight:700;
            color:#92400E; display:none; align-items:center; gap:8px;
        }
        .pop-indicator.show { display:flex; }

        /* ── FORMS / BUTTONS ── */
        .fg { margin-bottom:10px; }
        .fl { font-size:10px; font-weight:800; text-transform:uppercase; letter-spacing:0.7px; color:#475569; margin-bottom:5px; display:block; }
        .fi { width:100%; padding:9px 12px; border:2px solid #E2EAF8; border-radius:8px; font-size:13px; color:#0F172A; font-family:'Inter',sans-serif; outline:none; transition:border-color 0.2s,box-shadow 0.2s; }
        .fi:focus { border-color:#0077B6; box-shadow:0 0 0 3px rgba(0,119,182,0.1); }
        .frow { display:grid; grid-template-columns:1fr 1fr; gap:9px; }
        .btn { width:100%; padding:10px 14px; border-radius:9px; font-size:12px; font-weight:800; cursor:pointer; border:none; font-family:'Inter',sans-serif; transition:all 0.18s; }
        .btn:hover { transform:translateY(-1px); }
        .btn-blue  { background:linear-gradient(135deg,#003566,#0077B6); color:white; box-shadow:0 3px 12px rgba(0,119,182,0.35); }
        .btn-green { background:linear-gradient(135deg,#065F46,#06D6A0); color:white; box-shadow:0 3px 12px rgba(6,214,160,0.35); }
        .btn-red   { background:linear-gradient(135deg,#9B1C1C,#EF233C); color:white; box-shadow:0 3px 12px rgba(239,35,60,0.35); }
        .btn-ora   { background:linear-gradient(135deg,#92400E,#FFB703); color:white; box-shadow:0 3px 12px rgba(255,183,3,0.35); }
        .btn-gray  { background:#EEF4FF; color:#475569; border:2px solid #E2EAF8; }
        .btn-disch { background:linear-gradient(135deg,#4A044E,#A21CAF); color:white; box-shadow:0 3px 12px rgba(162,28,175,0.35); }
        .btn-sm { padding:8px 14px; font-size:11px; }
        .btn-xs { padding:6px 12px; font-size:11px; width:auto; }
        .bgrp { display:flex; gap:8px; }
        .divdr { border-top:2px solid #EEF4FF; margin:11px 0; }
        .sttl  { font-size:12px; font-weight:800; color:#334155; margin-bottom:3px; }
        .sdesc { font-size:11px; color:#64748B; margin-bottom:8px; }
        .simrow { display:flex; align-items:center; gap:7px; margin-top:8px; }
        .simdot { width:8px; height:8px; border-radius:50%; background:#CBD5E1; flex-shrink:0; }
        .simdot.on { background:#06D6A0; animation:blink 0.8s infinite; }
        .simtxt { font-size:11px; color:#64748B; }

        /* ── DISCHARGE SECTION ── */
        .discharge-box {
            background:#FAF5FF; border:2px solid #E9D5FF;
            border-radius:10px; padding:12px 14px; margin-top:2px;
        }
        .discharge-box .sttl { color:#7C3AED; }

        /* ── TABLE ── */
        .tbl-wrap { overflow-x:auto; }
        table { width:100%; border-collapse:collapse; font-size:12px; }
        th { background:#F1F5FF; padding:10px 12px; text-align:left; font-size:9px; text-transform:uppercase; letter-spacing:1.2px; font-weight:800; color:#64748B; border-bottom:2px solid #DBEAFE; white-space:nowrap; }
        td { padding:10px 12px; border-bottom:1px solid #F0F4FF; color:#1E293B; }
        tr:last-child td { border-bottom:none; }
        tr:hover td { background:#F8FAFF; }
        .pill { display:inline-block; padding:2px 8px; border-radius:20px; font-size:10px; font-weight:800; }
        .p-ok   { background:#D1FAE5; color:#065F46; }
        .p-warn { background:#FEF3C7; color:#92400E; }
        .p-crit { background:#FEE2E2; color:#9B1C1C; }
        .chg { font-weight:700; font-size:11px; }
        .chg-d { color:#EF233C; } .chg-u { color:#06D6A0; } .chg-n { color:#94A3B8; }

        /* ── CRISIS ── */
        #crisisSec { display:none; margin-top:17px; }
        .cr-card { background:white; border:3px solid #EF233C; border-radius:14px; padding:19px; box-shadow:0 6px 28px rgba(239,35,60,0.18); }
        .cr-hdr  { display:flex; justify-content:space-between; align-items:center; margin-bottom:11px; }
        .cr-ttl  { font-size:16px; font-weight:900; color:#9B1C1C; }
        .cr-sum  { background:#FFF0F0; border-radius:9px; padding:12px; margin-top:11px; border:1px solid #FECACA; }

        .ftr { text-align:center; padding:12px; background:white; border-top:2px solid #DBEAFE; font-size:11px; color:#94A3B8; }
        ::-webkit-scrollbar{width:5px} ::-webkit-scrollbar-track{background:#F0F4FF} ::-webkit-scrollbar-thumb{background:#BFDBFE;border-radius:3px}
    </style>
</head>
<body>

<!-- HEADER -->
<div class="hdr">
    <div>
        <div class="hdr-logo">SENTINEL ICU</div>
        <div class="hdr-title"><span class="hb">❤️</span> AI VITAL-STACK INTELLIGENCE SYSTEM</div>
        <div class="hdr-sub">50-Bed ICU · Real-Time Stack Monitoring · Predictive Trend Analysis · Deploy-Ready</div>
    </div>
    <div class="hdr-right">
        <canvas id="ecgCanvas" width="180" height="46" style="opacity:0.85;"></canvas>
        <div class="live-pill"><span class="live-dot"></span>LIVE &nbsp;<span id="clock">--:--:--</span></div>
        <div class="chip chip-ok" id="masterChip">● MONITORING</div>
    </div>
</div>

<!-- NAVIGATION -->
<div class="nav-bar">
    <button class="ntab active" id="tab-overview" onclick="showPage('overview')">🏥 ICU Overview</button>
    <button class="ntab"        id="tab-monitor"  onclick="showPage('monitor')">📊 Patient Monitor</button>
    <button class="ntab"        id="tab-reports"  onclick="showPage('reports')">📋 Reports</button>
</div>

<!-- INFO BAR -->
<div class="ibar">
    <span>ICU Ward: <span class="iv">50 Patient Beds</span></span>
    <span>Critical: <span class="iv-crit" id="ibCrit">—</span></span>
    <span>Warning:  <span class="iv-warn" id="ibWarn">—</span></span>
    <span>Stable:   <span class="iv-ok"   id="ibStab">—</span></span>
    <span>Vacant:   <span class="iv-vac"  id="ibVac">0</span></span>
    <span>Stack Mode: <span class="iv">LIFO · Max 100 items · Auto-POP on overflow</span></span>
    <span>Global Sim: <span class="iv" id="ibGSim" style="color:#EF233C;">STOPPED</span></span>
</div>

<!-- ALERT BANNER -->
<div id="alertBox"><span id="alertIcon" style="font-size:20px;"></span><span id="alertMsg"></span></div>

<!-- ═══════════════════════════════ PAGE 1: OVERVIEW ═══════════════════════════════ -->
<div class="page active" id="page-overview">
<div class="main">

    <div class="ward-stats">
        <div class="ws ws-total"><div class="ws-num" id="wsTot">50</div><div class="ws-lbl">Total Beds</div></div>
        <div class="ws ws-crit"><div class="ws-num" id="wsCrit">—</div><div class="ws-lbl">🔴 Critical</div></div>
        <div class="ws ws-warn"><div class="ws-num" id="wsWarn">—</div><div class="ws-lbl">🟡 Warning</div></div>
        <div class="ws ws-stab"><div class="ws-num" id="wsStab">—</div><div class="ws-lbl">🟢 Stable</div></div>
        <div class="ws ws-vac"><div class="ws-num" id="wsVac">0</div><div class="ws-lbl">🟦 Vacant</div></div>
    </div>

    <div class="ov-controls">
        <span class="oc-lbl">🌐 Global Simulator — all active beds:</span>
        <button class="btn btn-green btn-xs" onclick="startGlobalSim()">▶ Start All</button>
        <button class="btn btn-gray btn-xs"  onclick="stopGlobalSim()">⏹ Stop All</button>
        <div class="simrow">
            <div class="simdot" id="gSimDot"></div>
            <div class="simtxt" id="gSimTxt">Stopped — click Start All to simulate all active beds</div>
        </div>
    </div>

    <div class="beds-grid" id="bedsGrid">
        <div style="text-align:center;padding:40px;color:#94A3B8;grid-column:1/-1;">Loading 50 patient beds...</div>
    </div>
</div>
</div>

<!-- ═══════════════════════════════ PAGE 2: MONITOR ═══════════════════════════════ -->
<div class="page" id="page-monitor">
<div class="main">

    <div class="pat-sel-bar">
        <span class="psl">Monitoring:</span>
        <select class="psel" id="monSel" onchange="switchPatientMon(this.value)"><option>Loading...</option></select>
        <div class="psi">
            <span>Diagnosis: <span class="pv" id="monDiag">—</span></span>
            <span>Age: <span class="pv" id="monAge">—</span></span>
            <span>Stack: <span class="pv" id="monStack">0</span> items</span>
            <span>Last Push: <span class="pv" id="monTime">—</span></span>
        </div>
    </div>

    <div class="vcards">
        <div class="vc ok" id="cHR">
            <div class="vc-icon">❤️</div>
            <div class="vc-lbl">Heart Rate</div>
            <div class="vc-val" id="vHR">—</div>
            <div class="vc-unit">BPM (beats per minute)</div>
            <div class="vc-note" id="nHR">Awaiting data...</div>
        </div>
        <div class="vc ok" id="cBP">
            <div class="vc-icon">🩺</div>
            <div class="vc-lbl">Blood Pressure</div>
            <div class="vc-val md" id="vBP">—/—</div>
            <div class="vc-unit">mmHg — Systolic / Diastolic</div>
            <div class="vc-note" id="nBP">Awaiting data...</div>
        </div>
        <div class="vc ok" id="cO2">
            <div class="vc-icon">🫁</div>
            <div class="vc-lbl">Oxygen Saturation</div>
            <div class="vc-val" id="vO2">—</div>
            <div class="vc-unit">SpO₂ % (Normal: 95–100%)</div>
            <div class="vc-note" id="nO2">Awaiting data...</div>
        </div>
        <div class="vc" id="cRisk" style="--ac:#7C3AED;">
            <div class="vc-icon">🎯</div>
            <div class="vc-lbl">Risk Level — Trend Analysis</div>
            <div class="vc-val md" id="vRisk">—</div>
            <div class="vc-unit">10% drop rule · PEEK on top 3 items</div>
            <div class="vc-note" id="nRisk">Calculating...</div>
        </div>
        <div class="vc" style="--ac:#00B4D8;">
            <div class="vc-icon">📚</div>
            <div class="vc-lbl">Stack Memory</div>
            <div class="vc-val" id="vStack">0</div>
            <div class="vc-unit">Snapshots stored (Max: 100 · Auto-POP at limit)</div>
            <div class="vc-note" id="nStackMem" style="color:#0369A1;">LIFO Stack — Active</div>
        </div>
        <div class="vc" id="cTrend" style="--ac:#FFB703;">
            <div class="vc-icon">📈</div>
            <div class="vc-lbl">HR Rate of Change</div>
            <div class="vc-val md" id="vTrend">—</div>
            <div class="vc-unit">% change via Peek operation</div>
            <div class="vc-note" id="nTrend">Need 2+ readings...</div>
        </div>
    </div>

    <div class="twoCol">
        <!-- STACK VISUALIZATION -->
        <div class="sc">
            <div class="sc-ttl">📚 Stack Visualization — LIFO (Last In, First Out)</div>
            <div class="sthdr">
                <span class="top-badge">▲ TOP — PUSH goes here · POP takes from here</span>
                <span style="font-size:11px;color:#64748B;" id="stCount">0 items</span>
            </div>
            <div class="stlist" id="stList">
                <div style="text-align:center;padding:30px;color:#94A3B8;">Stack empty. Start simulator or push a reading.</div>
            </div>
            <div class="st-bot">▼ BOTTOM — oldest reading · auto-POPped first when stack hits 100</div>

            <!-- AUTO-POP INDICATOR -->
            <div class="pop-indicator" id="popIndicator">
                ⚡ <span id="popMsg">Auto-POP triggered — oldest reading removed to make room</span>
            </div>
        </div>

        <!-- CONTROLS -->
        <div class="sc">
            <div class="sc-ttl">⚡ Stack Controls</div>

            <div class="sttl">➕ Push New Reading Manually</div>
            <div style="height:7px;"></div>
            <div class="fg">
                <label class="fl">Heart Rate (BPM)</label>
                <input class="fi" type="number" id="iHR" placeholder="e.g. 78" min="30" max="220">
            </div>
            <div class="fg">
                <label class="fl">Blood Pressure (mmHg)</label>
                <div class="frow">
                    <input class="fi" type="number" id="iSys" placeholder="Systolic 120">
                    <input class="fi" type="number" id="iDia" placeholder="Diastolic 80">
                </div>
            </div>
            <div class="fg">
                <label class="fl">Oxygen Saturation (%)</label>
                <input class="fi" type="number" id="iO2" placeholder="e.g. 97" min="50" max="100">
            </div>
            <button class="btn btn-blue" onclick="pushReading()">➕ PUSH TO STACK</button>

            <div class="divdr"></div>
            <div class="sttl">🤖 Per-Patient Simulator</div>
            <div class="sdesc">Auto-pushes one reading every 5 sec for this patient</div>
            <div class="bgrp">
                <button class="btn btn-green" onclick="startSim()">▶ Start</button>
                <button class="btn btn-gray"  onclick="stopSim()">⏹ Stop</button>
            </div>
            <div class="simrow">
                <div class="simdot" id="simDot"></div>
                <div class="simtxt" id="simTxt">Stopped</div>
            </div>

            <div class="divdr"></div>
            <div class="sttl">🚨 Crisis Tools</div>
            <div class="sdesc">Simulate a crash or view emergency timeline</div>
            <div class="bgrp">
                <button class="btn btn-ora btn-sm" onclick="simCrash()">💀 Simulate Crash</button>
                <button class="btn btn-red btn-sm" onclick="goToReports()">🚨 Crisis Report</button>
            </div>

            <!-- DISCHARGE SECTION -->
            <div class="divdr"></div>
            <div class="discharge-box">
                <div class="sttl" style="color:#7C3AED;">🚪 Patient Discharge — Stack POP Operation</div>
                <div class="sdesc">Calls POP repeatedly until stack is empty. Marks bed as VACANT. Use when patient leaves the ICU.</div>
                <button class="btn btn-disch" onclick="dischargePatient()">🚪 Discharge Patient &amp; Clear Stack</button>
            </div>
        </div>
    </div>
</div>
</div>

<!-- ═══════════════════════════════ PAGE 3: REPORTS ═══════════════════════════════ -->
<div class="page" id="page-reports">
<div class="main">

    <div class="pat-sel-bar">
        <span class="psl">Reports for:</span>
        <select class="psel" id="repSel" onchange="switchPatientRep(this.value)"><option>Loading...</option></select>
        <div class="psi">
            <span>Diagnosis: <span class="pv" id="repDiag">—</span></span>
            <span>Total Readings: <span class="pv" id="repTotal">—</span></span>
        </div>
        <button class="btn btn-red btn-xs" onclick="openCrisis()">🚨 Generate Crisis Report</button>
    </div>

    <div class="sc" style="margin-bottom:17px;">
        <div class="sc-ttl">📜 Vital History — Stack Contents (Last 20, Newest First)</div>
        <div class="tbl-wrap">
            <table>
                <thead>
                    <tr><th>#</th><th>Timestamp</th><th>Heart Rate</th><th>Blood Pressure</th><th>SpO₂</th><th>HR Δ Change</th><th>Status</th></tr>
                </thead>
                <tbody id="repHistBody">
                    <tr><td colspan="7" style="text-align:center;padding:28px;color:#94A3B8;">Select a patient to view their history.</td></tr>
                </tbody>
            </table>
        </div>
    </div>

    <div id="crisisSec">
        <div class="cr-card">
            <div class="cr-hdr">
                <div class="cr-ttl">🚨 CRISIS REPORT — EMERGENCY TIMELINE</div>
                <button class="btn btn-gray btn-xs" onclick="closeCrisis()">✕ Close</button>
            </div>
            <div id="crisisBody"></div>
            <div class="cr-sum" id="crisisSum"></div>
        </div>
    </div>
</div>
</div>

<!-- FOOTER -->
<div class="ftr">
    SENTINEL ICU &nbsp;|&nbsp; DSA Project — Stack: PUSH · POP · PEEK &nbsp;|&nbsp;
    50-Bed ICU Ward · Auto-POP at 100-item limit · Patient Discharge POP &nbsp;|&nbsp;
    <span style="color:#EF233C;font-weight:900;">SAII</span>
    &nbsp;|&nbsp; Differential Trend Analysis
</div>

<script>
// ═══════════════ GLOBALS ═══════════════
let currentBed    = 'BED-01';
let currentPage   = 'overview';
let simInterval   = null;
let simRunning    = false;
let globalSimInt  = null;
let globalSimOn   = false;
let overviewInt   = null;
let monitorInt    = null;
let patCache      = {};
let prevAutoPops  = 0;   // Track auto-pop changes

// ═══════════════ ECG ANIMATION ═══════════════
(function(){
    const c=document.getElementById('ecgCanvas'); if(!c)return;
    const ctx=c.getContext('2d'); let off=0;
    function ecgY(x){const p=60,t=((x%p)+p)%p;if(t<8)return 23;else if(t<12)return 23-(t-8)/4*18;else if(t<14)return 5+(t-12)/2*36;else if(t<16)return 41-(t-14)/2*18;else if(t<20)return 23-(t-16)/4*5;else if(t<24)return 18+(t-20)/4*5;else return 23;}
    function draw(){ctx.clearRect(0,0,180,46);ctx.beginPath();ctx.strokeStyle='rgba(6,214,160,0.9)';ctx.lineWidth=1.8;for(let x=0;x<180;x++){const y=ecgY(x+off);x===0?ctx.moveTo(x,y):ctx.lineTo(x,y);}ctx.stroke();off=(off+1.2)%60;requestAnimationFrame(draw);}
    draw();
})();

// ═══════════════ CLOCK ═══════════════
setInterval(()=>{document.getElementById('clock').textContent=new Date().toLocaleTimeString();},1000);
document.getElementById('clock').textContent=new Date().toLocaleTimeString();

// ═══════════════ INIT ═══════════════
async function init(){
    try{
        const res=await fetch('/api/patients');
        const pts=await res.json();
        pts.forEach(p=>{patCache[p.bed_id]=p;});
        populateSelectors(pts);
        showPage('overview');
    }catch(e){console.error('Init failed:',e);}
}

function populateSelectors(patients){
    const opts=patients.map(p=>`<option value="${p.bed_id}">${p.bed_id} — ${p.name===''||p.name==='VACANT'?'[VACANT]':p.name} ${p.age&&p.age!=='—'?'('+p.age+'y)':''}</option>`).join('');
    ['monSel','repSel'].forEach(id=>{const el=document.getElementById(id);if(el)el.innerHTML=opts;});
}

// ═══════════════ NAVIGATION ═══════════════
function showPage(pageName){
    clearInterval(overviewInt); clearInterval(monitorInt);
    document.querySelectorAll('.page').forEach(p=>p.classList.remove('active'));
    document.getElementById('page-'+pageName).classList.add('active');
    document.querySelectorAll('.ntab').forEach(t=>t.classList.remove('active'));
    document.getElementById('tab-'+pageName).classList.add('active');
    document.getElementById('alertBox').className='';
    currentPage=pageName;
    if(pageName==='overview'){
        refreshOverview(); overviewInt=setInterval(refreshOverview,4000);
    }else if(pageName==='monitor'){
        syncSelectors(); updateMonitorHeader();
        refreshMonitor(); fetchStack();
        monitorInt=setInterval(()=>{refreshMonitor();fetchStack();},3000);
    }else if(pageName==='reports'){
        syncSelectors(); updateRepHeader(); fetchRepHistory();
        document.getElementById('crisisSec').style.display='none';
    }
}

function syncSelectors(){['monSel','repSel'].forEach(id=>{const el=document.getElementById(id);if(el)el.value=currentBed;});}
function switchPatientMon(bed){currentBed=bed;updateMonitorHeader();refreshMonitor();fetchStack();prevAutoPops=0;}
function switchPatientRep(bed){currentBed=bed;updateRepHeader();fetchRepHistory();document.getElementById('crisisSec').style.display='none';}
function viewBed(bedId){currentBed=bedId;showPage('monitor');}
function goToReports(){showPage('reports');}
function updateMonitorHeader(){const p=patCache[currentBed];if(p){document.getElementById('monDiag').textContent=p.diagnosis||'—';document.getElementById('monAge').textContent=p.age&&p.age!=='—'?p.age+'y':'—';}}
function updateRepHeader(){const p=patCache[currentBed];if(p)document.getElementById('repDiag').textContent=p.diagnosis||'—';}

// ═══════════════ OVERVIEW ═══════════════
async function refreshOverview(){
    try{
        const res=await fetch('/api/patients'); if(!res.ok)return;
        const pts=await res.json();
        let crit=0,warn=0,stab=0,vac=0;
        const grid=document.getElementById('bedsGrid');

        grid.innerHTML=pts.map(p=>{
            const s=p.status;

            // ── VACANT BED ──
            if(s.status==='VACANT'){
                vac++;
                return `<div class="bed-card vacant">
                    <div class="bc-hdr"><span class="bc-bed">${p.bed_id}</span><span class="bc-pill vacant">🟦 VACANT</span></div>
                    <div class="bc-name" style="color:#94A3B8;">Bed Available</div>
                    <div class="bc-diag" style="color:#CBD5E1;">Previously: ${s.prev_patient||'—'}</div>
                    <div style="padding:4px 5px;font-size:10px;color:#CBD5E1;margin-bottom:6px;">Stack cleared — all readings popped on discharge</div>
                </div>`;
            }

            // ── NO DATA ──
            if(s.status==='NO_DATA'){
                return `<div class="bed-card nodata">
                    <div class="bc-hdr"><span class="bc-bed">${p.bed_id}</span><span class="bc-pill nodata">NO DATA</span></div>
                    <div class="bc-name">${p.name}</div>
                    <div class="bc-diag">${p.diagnosis}</div>
                    <button class="btn btn-blue btn-xs" style="margin-left:5px;" onclick="viewBed('${p.bed_id}')">📊 Monitor</button>
                </div>`;
            }

            // ── ACTIVE BED ──
            const r=s.latest_reading, hra=s.trend_analysis.heart_rate_analysis, o2a=s.trend_analysis.oxygen_analysis;
            const risk=hra.risk_level;
            let cls='stable';
            if(risk===2||o2a.status==='CRITICAL'){cls='critical';crit++;}
            else if(risk===1||o2a.status==='WARNING'){cls='warning';warn++;}
            else stab++;
            const stTxt=cls==='critical'?'🔴 CRITICAL':cls==='warning'?'🟡 WARNING':'🟢 STABLE';
            const tStr=r.timestamp.includes(' ')?r.timestamp.split(' ')[1]:r.timestamp;

            return `<div class="bed-card ${cls}">
                <div class="bc-hdr"><span class="bc-bed">${p.bed_id}</span><span class="bc-pill ${cls}">${stTxt}</span></div>
                <div class="bc-name">${p.name}</div>
                <div class="bc-diag">${p.diagnosis} · ${p.age}y</div>
                <div class="bc-vitals">
                    <div class="bv"><span class="bv-icon">❤️</span><span class="bv-val ${r.heart_rate<65?'dng':''}">${r.heart_rate}</span><span class="bv-unit">BPM</span></div>
                    <div class="bv"><span class="bv-icon">🩺</span><span class="bv-val">${r.bp_systolic}/${r.bp_diastolic}</span><span class="bv-unit">mmHg</span></div>
                    <div class="bv"><span class="bv-icon">🫁</span><span class="bv-val ${r.oxygen<90?'dng':''}">${r.oxygen}%</span><span class="bv-unit">SpO₂</span></div>
                    <div class="bv"><span class="bv-icon">📚</span><span class="bv-val">${s.stack_size}</span><span class="bv-unit">Stack</span></div>
                </div>
                <div class="bc-time">Last: ${tStr}</div>
                <button class="btn btn-blue btn-xs" style="margin-left:5px;" onclick="viewBed('${p.bed_id}')">📊 View Monitor →</button>
            </div>`;
        }).join('');

        document.getElementById('wsTot').textContent=pts.length;
        document.getElementById('wsCrit').textContent=crit;
        document.getElementById('wsWarn').textContent=warn;
        document.getElementById('wsStab').textContent=stab;
        document.getElementById('wsVac').textContent=vac;
        document.getElementById('ibCrit').textContent=crit;
        document.getElementById('ibWarn').textContent=warn;
        document.getElementById('ibStab').textContent=stab;
        document.getElementById('ibVac').textContent=vac;

        const chip=document.getElementById('masterChip');
        if(crit>0){chip.className='chip chip-crit';chip.textContent='🚨 '+crit+' CRITICAL';}
        else if(warn>0){chip.className='chip chip-warn';chip.textContent='⚠️ '+warn+' WARNING';}
        else{chip.className='chip chip-ok';chip.textContent='● ALL MONITORING';}

        const ovTab=document.getElementById('tab-overview');
        if(crit>0)ovTab.innerHTML=`🏥 ICU Overview <span class="nbadge">${crit}</span>`;
        else ovTab.textContent='🏥 ICU Overview';

    }catch(e){}
}

// ═══════════════ MONITOR ═══════════════
async function refreshMonitor(){
    try{
        const res=await fetch(`/api/status?bed=${currentBed}`); if(!res.ok)return;
        const data=await res.json();
        if(data.status==='VACANT'){renderVacantMonitor(data);return;}
        if(data.status==='NO_DATA') return;
        renderCards(data); renderAlert(data);
    }catch(e){}
}

function renderVacantMonitor(data){
    document.getElementById('vHR').textContent='—';
    document.getElementById('vBP').textContent='—/—';
    document.getElementById('vO2').textContent='—';
    document.getElementById('vRisk').textContent='VACANT';
    document.getElementById('vStack').textContent='0';
    document.getElementById('vTrend').textContent='—';
    document.getElementById('monStack').textContent='0';
    document.getElementById('nHR').textContent='Bed vacant — patient discharged';
    document.getElementById('nBP').textContent='—';
    document.getElementById('nO2').textContent='—';
    document.getElementById('nRisk').textContent='Stack fully cleared via discharge POP';
    document.getElementById('nTrend').textContent='—';
    document.getElementById('nStackMem').textContent='Stack cleared by discharge';
    document.getElementById('nStackMem').style.color='#94A3B8';
    document.getElementById('stList').innerHTML='<div style="text-align:center;padding:30px;color:#94A3B8;">📭 Stack is empty — patient was discharged.<br><small>All readings were popped from the stack.</small></div>';
    document.getElementById('stCount').textContent='0 items';
    document.getElementById('alertBox').className='';
    document.getElementById('popIndicator').classList.remove('show');
    ['cHR','cBP','cO2'].forEach(id=>{const c=document.getElementById(id);if(c)c.className='vc ok';});
    const rc=document.getElementById('cRisk');
    if(rc){rc.className='vc';rc.style.setProperty('--ac','#94A3B8');}
}

function renderCards(data){
    const r=data.latest_reading, hra=data.trend_analysis.heart_rate_analysis, o2a=data.trend_analysis.oxygen_analysis;

    document.getElementById('monStack').textContent=data.stack_size;
    document.getElementById('vStack').textContent=data.stack_size;
    document.getElementById('stCount').textContent=data.stack_size+' items';
    document.getElementById('monTime').textContent=new Date().toLocaleTimeString();

    // Auto-POP indicator
    const autoPops = data.auto_pops || 0;
    const stackNote = document.getElementById('nStackMem');
    const popInd    = document.getElementById('popIndicator');
    if(autoPops > 0){
        stackNote.textContent=`⚡ Auto-popped ${autoPops}× at 100-item limit`;
        stackNote.style.color='#92400E';
        if(autoPops > prevAutoPops){
            // New auto-pop just happened
            document.getElementById('popMsg').textContent=
                `Auto-POP #${autoPops}: oldest reading removed — stack was at 100-item limit`;
            popInd.classList.add('show');
            setTimeout(()=>popInd.classList.remove('show'), 4000);
        }
    } else {
        stackNote.textContent='LIFO Stack — Active (auto-POP triggers at 100 items)';
        stackNote.style.color='#0369A1';
        popInd.classList.remove('show');
    }
    prevAutoPops = autoPops;

    document.getElementById('vHR').textContent=r.heart_rate;
    applyCard('cHR','nHR',hra.status,
        hra.status==='CRITICAL'?'🚨 Critical drop — intervene NOW':
        hra.status==='WARNING' ?'⚠️ Declining — monitor closely':'✓ Stable');

    document.getElementById('vBP').textContent=r.bp_systolic+'/'+r.bp_diastolic;
    const bpS=r.bp_systolic>140?'CRITICAL':r.bp_systolic>130?'WARNING':'NORMAL';
    applyCard('cBP','nBP',bpS,bpS==='CRITICAL'?'🚨 Hypertension!':bpS==='WARNING'?'⚠️ Elevated':'✓ Normal range');

    document.getElementById('vO2').textContent=r.oxygen;
    applyCard('cO2','nO2',o2a.status,
        o2a.status==='CRITICAL'?'🚨 Hypoxia! Immediate action!':
        o2a.status==='WARNING' ?'⚠️ Below normal range':'✓ Good oxygenation');

    const risk=hra.risk_level, rc=document.getElementById('cRisk');
    if(risk===2){document.getElementById('vRisk').textContent='CRITICAL';rc.className='vc crit';rc.style.setProperty('--ac','#EF233C');document.getElementById('nRisk').textContent='🚨 PRE-CRASH ALERT!';}
    else if(risk===1){document.getElementById('vRisk').textContent='WARNING';rc.className='vc warn';rc.style.setProperty('--ac','#FFB703');document.getElementById('nRisk').textContent='⚠️ Trend detected';}
    else{document.getElementById('vRisk').textContent='LOW';rc.className='vc ok';rc.style.setProperty('--ac','#06D6A0');document.getElementById('nRisk').textContent='✓ No dangerous trends';}

    const changes=hra.changes||[];
    if(changes.length>0){
        const last=changes[changes.length-1];
        document.getElementById('vTrend').textContent=(last>=0?'+':'')+last.toFixed(1)+'%';
        const tc=document.getElementById('cTrend');
        if(last<=-10){tc.style.setProperty('--ac','#EF233C');document.getElementById('nTrend').textContent='⬇️ Critical drop (≥10%)';}
        else if(last<-2){tc.style.setProperty('--ac','#FFB703');document.getElementById('nTrend').textContent='⬇️ Slight decrease';}
        else{tc.style.setProperty('--ac','#06D6A0');document.getElementById('nTrend').textContent=last>0?'⬆️ Rising':'→ Stable';}
    }
}

function applyCard(cid,nid,status,note){
    const c=document.getElementById(cid),n=document.getElementById(nid);
    if(c)c.className=status==='CRITICAL'?'vc crit':status==='WARNING'?'vc warn':'vc ok';
    if(n)n.textContent=note;
}

function renderAlert(data){
    const box=document.getElementById('alertBox'),hra=data.trend_analysis.heart_rate_analysis,o2a=data.trend_analysis.oxygen_analysis;
    if(hra.risk_level===2||o2a.status==='CRITICAL'){box.className='ac';document.getElementById('alertIcon').textContent='🚨';document.getElementById('alertMsg').textContent=hra.risk_level===2?hra.message:o2a.message;}
    else if(hra.risk_level===1||o2a.status==='WARNING'){box.className='aw';document.getElementById('alertIcon').textContent='⚠️';document.getElementById('alertMsg').textContent=hra.risk_level===1?hra.message:o2a.message;}
    else box.className='';
}

// ═══════════════ STACK VISUALIZATION ═══════════════
async function fetchStack(){
    try{
        const res=await fetch(`/api/stack-items?bed=${currentBed}`); if(!res.ok)return;
        const items=await res.json();
        const list=document.getElementById('stList');
        if(!items||items.length===0){list.innerHTML='<div style="text-align:center;padding:30px;color:#94A3B8;">Stack empty. Start simulator or push a reading.</div>';return;}
        const show=items.slice(-6).reverse(),total=items.length;
        list.innerHTML=show.map((item,i)=>{
            const isTop=i===0,tStr=item.timestamp.includes(' ')?item.timestamp.split(' ')[1]:item.timestamp;
            return `<div class="si ${isTop?'top':''}">
                <div class="si-vitals">
                    <div><div class="sv-lbl">HR</div><div class="sv-val ${item.heart_rate<65?'r':''}">${item.heart_rate} BPM</div></div>
                    <div><div class="sv-lbl">BP</div><div class="sv-val">${item.bp_systolic}/${item.bp_diastolic}</div></div>
                    <div><div class="sv-lbl">SpO₂</div><div class="sv-val ${item.oxygen<90?'r':''}">${item.oxygen}%</div></div>
                    <div><div class="sv-lbl">Time</div><div class="sv-val" style="font-size:11px;">${tStr}</div></div>
                </div>
                <span class="si-badge ${isTop?'b-top':'b-num'}">${isTop?'▲ TOP':'#'+(total-i)}</span>
            </div>`;
        }).join('')+(total>6?`<div style="text-align:center;font-size:11px;color:#94A3B8;padding:5px;">…${total-6} more below</div>`:'');
    }catch(e){}
}

// ═══════════════ PUSH ═══════════════
async function pushReading(){
    const hr=parseInt(document.getElementById('iHR').value),sys=parseInt(document.getElementById('iSys').value),dia=parseInt(document.getElementById('iDia').value),o2=parseInt(document.getElementById('iO2').value);
    if([hr,sys,dia,o2].some(isNaN)){alert('⚠️ Please fill all four fields.');return;}
    if(hr<30||hr>220){alert('Heart rate: 30–220');return;}
    if(sys<60||sys>250){alert('Systolic: 60–250');return;}
    if(dia<30||dia>150){alert('Diastolic: 30–150');return;}
    if(o2<50||o2>100){alert('Oxygen: 50–100%');return;}
    try{
        const res=await fetch('/api/add-reading',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({bed:currentBed,heart_rate:hr,bp_systolic:sys,bp_diastolic:dia,oxygen:o2})});
        const d=await res.json();
        if(d.success){['iHR','iSys','iDia','iO2'].forEach(id=>document.getElementById(id).value='');refreshMonitor();fetchStack();}
        else alert('Error: '+(d.error||'Unknown error'));
    }catch(e){alert('Server error.');}
}

// ═══════════════ SIMULATORS ═══════════════
function startSim(){
    if(simRunning)return;simRunning=true;
    document.getElementById('simDot').classList.add('on');
    document.getElementById('simTxt').textContent='Running for '+currentBed+' — every 5 sec';
    doSim();simInterval=setInterval(doSim,5000);
}
function stopSim(){simRunning=false;clearInterval(simInterval);document.getElementById('simDot').classList.remove('on');document.getElementById('simTxt').textContent='Stopped';}
async function doSim(){try{await fetch('/api/simulate',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({bed:currentBed})});refreshMonitor();fetchStack();}catch(e){}}

function startGlobalSim(){
    if(globalSimOn)return;globalSimOn=true;
    document.getElementById('gSimDot').classList.add('on');
    document.getElementById('gSimTxt').textContent='Running — updating all active beds every 5 seconds';
    document.getElementById('ibGSim').textContent='ACTIVE';document.getElementById('ibGSim').style.color='#065F46';
    doGlobalSim();globalSimInt=setInterval(doGlobalSim,5000);
}
function stopGlobalSim(){
    globalSimOn=false;clearInterval(globalSimInt);
    document.getElementById('gSimDot').classList.remove('on');
    document.getElementById('gSimTxt').textContent='Stopped';
    document.getElementById('ibGSim').textContent='STOPPED';document.getElementById('ibGSim').style.color='#EF233C';
}
async function doGlobalSim(){try{await fetch('/api/simulate-all',{method:'POST'});if(currentPage==='overview')refreshOverview();else if(currentPage==='monitor'){refreshMonitor();fetchStack();}}catch(e){}}

// ═══════════════ SIMULATE CRASH ═══════════════
async function simCrash(){
    if(!confirm(`Push 3 declining readings for ${currentBed} to trigger Pre-Crash Alert?\n\nDemonstrates the Differential Trend Analyzer detecting 3 consecutive ≥10% HR drops.`))return;
    try{await fetch('/api/simulate-crisis',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({bed:currentBed})});refreshMonitor();fetchStack();}catch(e){}
}

// ═══════════════ DISCHARGE (POP ALL) ═══════════════
async function dischargePatient(){
    const p=patCache[currentBed];
    const pName=p?p.name:'this patient';
    const stackSize=document.getElementById('vStack').textContent||'?';

    if(pName==='VACANT'||pName===''){alert('This bed is already vacant.');return;}

    if(!confirm(`🚪 Discharge ${pName} from ${currentBed}?\n\nThis will:\n  • POP all ${stackSize} readings from the stack\n  • Clear the entire vital history\n  • Mark the bed as VACANT\n\nThis operation cannot be undone.`))return;

    try{
        const res=await fetch('/api/discharge',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({bed:currentBed})});
        const data=await res.json();
        if(data.success){
            // Update local cache
            if(patCache[currentBed]){
                patCache[currentBed].name='VACANT';
                patCache[currentBed].diagnosis='Bed Available';
            }
            // Update selector options
            const selOpts=document.querySelectorAll(`#monSel option[value="${currentBed}"], #repSel option[value="${currentBed}"]`);
            selOpts.forEach(opt=>{opt.textContent=`${currentBed} — [VACANT]`;});

            alert(`✅ Discharge Complete\n\nPatient: ${data.patient}\nBed: ${data.bed_id}\nReadings POPped from stack: ${data.readings_popped}\nStatus: BED IS NOW VACANT`);
            prevAutoPops=0;
            showPage('overview');
        }else{
            alert('Error: '+(data.error||'Discharge failed'));
        }
    }catch(e){alert('Server error during discharge.');}
}

// ═══════════════ REPORTS ═══════════════
async function fetchRepHistory(){
    try{
        const res=await fetch(`/api/history?bed=${currentBed}&n=20`); if(!res.ok)return;
        const items=await res.json();
        const tbody=document.getElementById('repHistBody');
        document.getElementById('repTotal').textContent=items.length+' readings';
        if(!items||items.length===0){tbody.innerHTML='<tr><td colspan="7" style="text-align:center;padding:28px;color:#94A3B8;">No readings. Bed may be vacant.</td></tr>';return;}
        const rev=[...items].reverse();
        tbody.innerHTML=rev.map((r,i)=>{
            const prev=rev[i+1];
            let chgHtml='<span class="chg chg-n">—</span>';
            if(prev){const pct=((r.heart_rate-prev.heart_rate)/prev.heart_rate*100);chgHtml=`<span class="chg ${pct<=-5?'chg-d':pct>0?'chg-u':'chg-n'}">${pct>=0?'+':''}${pct.toFixed(1)}%</span>`;}
            const pill=r.heart_rate<60||r.oxygen<88?'p-crit':r.heart_rate<70||r.oxygen<93?'p-warn':'p-ok';
            return `<tr>
                <td style="color:#94A3B8;font-size:11px;">${rev.length-i}</td>
                <td style="font-size:11px;">${r.timestamp}</td>
                <td style="font-weight:800;color:${r.heart_rate<65?'#EF233C':'#0F172A'};">${r.heart_rate} BPM</td>
                <td>${r.bp_systolic}/${r.bp_diastolic} mmHg</td>
                <td style="font-weight:800;color:${r.oxygen<90?'#EF233C':'#0F172A'};">${r.oxygen}%</td>
                <td>${chgHtml}</td>
                <td><span class="pill ${pill}">${pill==='p-crit'?'Critical':pill==='p-warn'?'Warning':'Normal'}</span></td>
            </tr>`;
        }).join('');
    }catch(e){}
}

async function openCrisis(){
    try{
        const res=await fetch(`/api/crisis-report?bed=${currentBed}`);
        const data=await res.json();
        if(!data.readings||data.readings.length===0){alert('No readings available for crisis report. Bed may be vacant.');return;}
        const rows=data.readings.map((r,i)=>{
            const prev=data.readings[i-1];
            let chg='—',cc='#0F172A';
            if(prev){const pct=((r.heart_rate-prev.heart_rate)/prev.heart_rate*100);chg=(pct>=0?'+':'')+pct.toFixed(1)+'%';cc=pct<=-10?'#EF233C':pct<-3?'#FFB703':'#06D6A0';}
            return `<tr><td style="font-weight:800;color:#9B1C1C;">T−${data.readings.length-i}</td><td style="font-size:11px;">${r.timestamp}</td><td style="font-weight:800;color:${r.heart_rate<70?'#EF233C':'#0F172A'};">${r.heart_rate} BPM</td><td>${r.bp_systolic}/${r.bp_diastolic}</td><td style="font-weight:800;color:${r.oxygen<90?'#EF233C':'#0F172A'};">${r.oxygen}%</td><td style="font-weight:700;color:${cc};">${chg}</td></tr>`;
        }).join('');
        document.getElementById('crisisBody').innerHTML=`<p style="font-size:12px;color:#9B1C1C;margin-bottom:10px;">${new Date().toLocaleString()} · ${currentBed} · ${data.readings.length} readings</p><div class="tbl-wrap"><table><thead><tr><th>Timeline</th><th>Timestamp</th><th>Heart Rate</th><th>Blood Pressure</th><th>SpO₂</th><th>HR Δ</th></tr></thead><tbody>${rows}</tbody></table></div>`;
        document.getElementById('crisisSum').innerHTML=`<strong style="color:#9B1C1C;font-size:12px;">📋 Analysis:</strong><br><span style="font-size:12px;line-height:1.65;">${data.summary}</span>`;
        const sec=document.getElementById('crisisSec');sec.style.display='block';sec.scrollIntoView({behavior:'smooth'});
    }catch(e){alert('Error generating report.');}
}
function closeCrisis(){document.getElementById('crisisSec').style.display='none';}

// ═══════════════ START ═══════════════
init();
</script>
</body>
</html>"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)

print("✓ HTML updated: discharge button, auto-pop indicator, vacant bed handling")
print("✓ POP operation now visible in UI with real-time notifications")

✓ HTML updated: discharge button, auto-pop indicator, vacant bed handling
✓ POP operation now visible in UI with real-time notifications


In [6]:
# ================================================================
# CELL 3: Flask backend — All 3 POP operations explicitly shown
#   1. pop()         → remove top (newest) reading
#   2. pop_oldest()  → remove bottom (oldest) reading
#   3. pop_all()     → discharge: remove ALL readings from stack
# ================================================================

from flask import Flask, jsonify, render_template, request
from flask_cors import CORS
import random, os, csv
from datetime import datetime


# ════════════════════════════════════════════════════════════════
# DATA STRUCTURE 1: Medical Snapshot — one node in the Stack
# ════════════════════════════════════════════════════════════════
class MedicalSnapshot:
    def __init__(self, timestamp, heart_rate, bp_systolic, bp_diastolic, oxygen):
        self.timestamp    = str(timestamp)
        self.heart_rate   = int(heart_rate)
        self.bp_systolic  = int(bp_systolic)
        self.bp_diastolic = int(bp_diastolic)
        self.oxygen       = int(oxygen)

    def to_dict(self):
        return {'timestamp':self.timestamp,'heart_rate':self.heart_rate,
                'bp_systolic':self.bp_systolic,'bp_diastolic':self.bp_diastolic,
                'oxygen':self.oxygen}


# ════════════════════════════════════════════════════════════════
# DATA STRUCTURE 2: Vital Stack — LIFO with 3 POP variants
# ════════════════════════════════════════════════════════════════
class VitalStack:
    """
    LIFO Stack storing MedicalSnapshot objects.

    PUSH  — add new reading to TOP               (O(1))
    POP   — remove newest reading from TOP        (O(1))  ← standard LIFO
    PEEK  — view top N items without removing     (O(1))  ← used for trend analysis
    SIZE  — count of items in stack               (O(1))

    Auto-POP: when stack hits max_size (100), the oldest
              reading is automatically popped before pushing.
              Counter tracked in self.auto_pops.

    pop_oldest() — explicitly remove the oldest (bottom) reading
    pop_all()    — remove ALL readings (used for patient discharge)
    """
    def __init__(self, max_size=100):
        self._items    = []         # internal list; index[-1] = TOP of stack
        self.max_size  = max_size
        self.auto_pops = 0          # count of automatic pops at 100-item limit

    def push(self, snapshot):
        """
        PUSH operation — O(1).
        If stack is at 100-item capacity, automatically POPs the
        oldest reading first to make room (circular buffer behaviour).
        Returns the auto-popped item, or None if no auto-pop was needed.
        """
        auto_popped = None
        if len(self._items) >= self.max_size:
            auto_popped   = self._items.pop(0)  # POP oldest (bottom of stack)
            self.auto_pops += 1
        self._items.append(snapshot)
        return auto_popped   # caller can inspect what was dropped

    def pop(self):
        """
        POP operation — O(1). Standard LIFO.
        Removes and returns the TOP (most recent) reading.
        Returns None if stack is empty.
        """
        return self._items.pop() if self._items else None

    def pop_oldest(self):
        """
        POP from bottom — O(1).
        Removes and returns the OLDEST reading.
        Used for explicit cleanup of stale data.
        Returns None if stack is empty.
        """
        return self._items.pop(0) if self._items else None

    def pop_all(self):
        """
        POP ALL — O(n).
        Removes every reading from the stack.
        Used when a patient is discharged from the ICU.
        Returns the count of readings that were cleared.
        """
        count = len(self._items)
        self._items.clear()
        return count   # caller logs how many were removed

    def peek(self, offset=0):
        """
        PEEK — O(1). Does NOT remove the item.
        offset=0 → TOP (newest)
        offset=1 → one below top
        offset=2 → two below top
        """
        if not self._items or offset >= len(self._items):
            return None
        return self._items[-(offset + 1)]

    def is_empty(self):    return len(self._items) == 0
    def size(self):        return len(self._items)

    def get_top_n(self, n):
        """Return last N items as list (oldest→newest) for trend analysis."""
        return self._items[-n:] if n <= len(self._items) else list(self._items)

    def get_all(self):
        return [s.to_dict() for s in self._items]


# ════════════════════════════════════════════════════════════════
# ALGORITHM: Differential Trend Analyzer
# ════════════════════════════════════════════════════════════════
class TrendAnalyzer:
    """
    Uses PEEK to look at top 3 stack items WITHOUT removing them.
    Triggers PRE-CRASH ALERT if HR drops ≥10% in 2 consecutive readings.
    """
    def __init__(self, stack):
        self.stack = stack

    def _pct(self, old, new):
        return 0.0 if old == 0 else ((new - old) / old) * 100.0

    def analyze_heart_rate(self):
        result = {'status':'NORMAL','message':'Heart rate stable',
                  'hr_values':[],'changes':[],'risk_level':0}
        recent = self.stack.get_top_n(3)
        if len(recent) < 2:
            return result
        hr_vals = [s.heart_rate for s in recent]
        result['hr_values'] = hr_vals
        changes = [self._pct(hr_vals[i], hr_vals[i+1]) for i in range(len(hr_vals)-1)]
        result['changes'] = changes
        critical = sum(1 for c in changes if c <= -10)
        moderate = sum(1 for c in changes if c <  -3)
        if critical >= 2:
            result.update({'status':'CRITICAL','risk_level':2,
                'message':'🚨 PRE-CRASH ALERT: HR dropped ≥10% in last 2 consecutive readings!'})
        elif critical == 1 or moderate >= 2:
            result.update({'status':'WARNING','risk_level':1,
                'message':'⚠️ WARNING: Heart rate declining pattern detected.'})
        return result

    def analyze_oxygen(self):
        result = {'status':'NORMAL','message':'Oxygen normal','o2_values':[]}
        recent = self.stack.get_top_n(3)
        if not recent:
            return result
        o2_vals = [s.oxygen for s in recent]
        result['o2_values'] = o2_vals
        cur = o2_vals[-1]
        if cur < 85:
            result.update({'status':'CRITICAL',
                'message':f'🚨 CRITICAL HYPOXIA: SpO₂={cur}% (Danger threshold <85%!)'})
        elif cur < 92:
            result.update({'status':'WARNING',
                'message':f'⚠️ LOW OXYGEN: SpO₂={cur}% (Normal: 95–100%)'})
        return result


# ════════════════════════════════════════════════════════════════
# Per-Patient Tracker — owns one Stack + one Analyzer
# ════════════════════════════════════════════════════════════════
class ICUVitalTracker:
    def __init__(self, bed_id):
        self.bed_id   = bed_id
        self.stack    = VitalStack(max_size=100)
        self.analyzer = TrendAnalyzer(self.stack)
        self.csv_path = f'sensor_{bed_id}.csv'
        self._init_csv()

    def _init_csv(self):
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['timestamp','heart_rate','bp_systolic','bp_diastolic','oxygen'])

    def add_reading(self, hr, bp_sys, bp_dia, o2, ts=None):
        ts   = ts or datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        snap = MedicalSnapshot(ts, hr, bp_sys, bp_dia, o2)
        auto_popped = self.stack.push(snap)   # push() returns auto-popped item
        # File streaming — append to CSV
        with open(self.csv_path, 'a', newline='') as f:
            csv.writer(f).writerow([ts, hr, bp_sys, bp_dia, o2])
        return snap, auto_popped

    def discharge_patient(self):
        """
        Patient discharge — calls pop_all() to clear the entire stack.
        In a real ICU, this is triggered when the patient leaves the ward.
        Returns the number of readings that were cleared.
        """
        return self.stack.pop_all()   # POP ALL operation

    def get_status(self):
        latest = self.stack.peek(0)   # PEEK — does not remove
        if latest is None:
            return {'status': 'NO_DATA'}
        return {
            'status':         'OK',
            'latest_reading': latest.to_dict(),
            'stack_size':     self.stack.size(),
            'auto_pops':      self.stack.auto_pops,  # how many times auto-pop fired
            'trend_analysis': {
                'heart_rate_analysis': self.analyzer.analyze_heart_rate(),
                'oxygen_analysis':     self.analyzer.analyze_oxygen()
            }
        }

    def get_history(self, n=20):
        return [s.to_dict() for s in self.stack.get_top_n(n)]

    def get_crisis_report(self):
        readings = self.get_history(10)
        return {'readings': readings, 'summary': self._summary(readings)}

    def _summary(self, readings):
        if len(readings) < 2:
            return "Insufficient data for automated analysis."
        first_hr, last_hr = readings[0]['heart_rate'], readings[-1]['heart_rate']
        total = ((last_hr - first_hr) / first_hr) * 100
        drops = sum(1 for i in range(1, len(readings))
                    if readings[i]['heart_rate'] < readings[i-1]['heart_rate'])
        min_o2 = min(r['oxygen'] for r in readings)
        lines = []
        if total <= -15: lines.append(f"🚨 SEVERE DECLINE: HR fell {abs(total):.1f}% over {len(readings)} readings.")
        elif total <= -5: lines.append(f"⚠️ DECLINING: HR dropped {abs(total):.1f}% over {len(readings)} readings.")
        else:             lines.append(f"✓ HR stable (Δ{total:+.1f}% over {len(readings)} readings).")
        lines.append(f"Declining steps: {drops}/{len(readings)-1} transitions.")
        if min_o2 < 88:   lines.append(f"🚨 Critical SpO₂ recorded: {min_o2}%.")
        elif min_o2 < 94: lines.append(f"⚠️ Low SpO₂ recorded: {min_o2}%.")
        else:             lines.append(f"SpO₂ ≥{min_o2}% maintained throughout.")
        return " ".join(lines)


# ════════════════════════════════════════════════════════════════
# ICU Ward — manages all 50 beds
# ════════════════════════════════════════════════════════════════
class ICUWard:
    def __init__(self):
        self.beds = {}

    def add_bed(self, bed_id, name, age, diagnosis):
        self.beds[bed_id] = {
            'info':      {'name':name, 'age':age, 'diagnosis':diagnosis,
                          'admitted':datetime.now().strftime('%Y-%m-%d %H:%M'),
                          'vacant':False},
            'tracker':   ICUVitalTracker(bed_id),
            'sim_state': {'heart_rate':80,'bp_systolic':120,'bp_diastolic':80,'oxygen':97}
        }

    def get_bed(self, bed_id):
        return self.beds.get(bed_id)

    def all_statuses(self):
        result = []
        for bed_id, bed in self.beds.items():
            if bed['info'].get('vacant', False):
                result.append({
                    'bed_id':    bed_id,
                    'name':      'VACANT',
                    'age':       '—',
                    'diagnosis': 'Bed Available for New Patient',
                    'status': {
                        'status':       'VACANT',
                        'prev_patient': bed['info'].get('prev_patient', '—'),
                        'discharged_at':bed['info'].get('discharged_at', '—'),
                        'stack_size':   0,
                        'auto_pops':    bed['tracker'].stack.auto_pops
                    }
                })
            else:
                status = bed['tracker'].get_status()
                result.append({
                    'bed_id':    bed_id,
                    'name':      bed['info']['name'],
                    'age':       bed['info']['age'],
                    'diagnosis': bed['info']['diagnosis'],
                    'status':    status
                })
        return result

    def simulate_bed(self, bed_id):
        bed = self.get_bed(bed_id)
        if not bed or bed['info'].get('vacant', False):
            return False    # Skip vacant beds automatically
        nv = _gen_vitals(bed['sim_state'])
        bed['sim_state'] = nv
        bed['tracker'].add_reading(
            nv['heart_rate'], nv['bp_systolic'], nv['bp_diastolic'], nv['oxygen'])
        return True

    def simulate_all(self):
        for bed_id in self.beds:
            self.simulate_bed(bed_id)   # already skips vacant beds


# ════════════════════════════════════════════════════════════════
# Initial vitals by patient category
# ════════════════════════════════════════════════════════════════
def get_initial_vitals(category):
    if   category == 'CRIT_O2': return [(80,122,82,84),(79,120,80,83),(78,118,78,82)]
    elif category == 'CRIT_HR': return [(90,138,90,93),(81,135,88,92),(72,132,86,91)]
    elif category == 'CRIT_BP': return [(88,162,108,93),(87,165,110,93),(86,168,112,92)]
    elif category == 'WARN_HR': return [(95,142,94,92),(94,140,93,92),(93,138,92,91)]
    elif category == 'WARN_O2': return [(82,128,85,89),(81,127,84,88),(80,126,83,88)]
    elif category == 'WARN_BP': return [(85,148,98,94),(84,150,99,94),(83,152,100,93)]
    else:  # STABLE — randomized for variety
        hr  = random.randint(68, 84)
        sys = random.randint(112, 132)
        dia = random.randint(72, 88)
        o2  = random.randint(95, 99)
        def c(v,lo,hi): return max(lo, min(hi, v))
        return [
            (hr, sys, dia, o2),
            (c(hr+random.randint(-2,2),50,130), c(sys+random.randint(-2,2),85,170),
             c(dia+random.randint(-1,1),50,110), c(o2+random.randint(-1,1),82,100)),
            (c(hr+random.randint(-2,2),50,130), c(sys+random.randint(-2,2),85,170),
             c(dia+random.randint(-1,1),50,110), c(o2+random.randint(-1,1),82,100)),
        ]


# ════════════════════════════════════════════════════════════════
# 50 Patient Definitions
# ════════════════════════════════════════════════════════════════
WARD_PATIENTS = [
    ('BED-01','James Anderson',64,'Cardiac Arrest Recovery','CRIT_O2'),
    ('BED-02','Maria Rodriguez',71,'Respiratory Failure','CRIT_HR'),
    ('BED-03','Robert Thompson',55,'Septic Shock','CRIT_O2'),
    ('BED-04','Sarah Mitchell',78,'Acute Liver Failure','CRIT_HR'),
    ('BED-05','David Patel',62,'Pulmonary Embolism','CRIT_O2'),
    ('BED-06','Jennifer Lee',85,'Multi-Organ Failure','CRIT_HR'),
    ('BED-07','Michael Brown',59,'Traumatic Brain Injury','CRIT_BP'),
    ('BED-08','Lisa Garcia',43,'Cardiogenic Shock','CRIT_O2'),
    ('BED-09','William Chen',77,'Acute Kidney Injury','CRIT_HR'),
    ('BED-10','Patricia Wilson',68,'Hypertensive Emergency','CRIT_BP'),
    ('BED-11','Christopher Davis',52,'Severe Pneumonia','WARN_HR'),
    ('BED-12','Barbara Martinez',67,'Heart Failure Exacerbation','WARN_O2'),
    ('BED-13','Matthew Johnson',48,'Acute Stroke Recovery','WARN_HR'),
    ('BED-14','Susan Taylor',73,'COPD Exacerbation','WARN_O2'),
    ('BED-15','Anthony White',61,'Uncontrolled Hypertension','WARN_BP'),
    ('BED-16','Dorothy Harris',82,'Hip Fracture Complications','WARN_HR'),
    ('BED-17','Kevin Clark',57,'GI Bleeding','WARN_BP'),
    ('BED-18','Helen Lewis',69,'Acute Pancreatitis','WARN_O2'),
    ('BED-19','Brian Robinson',44,'Spinal Cord Injury','WARN_HR'),
    ('BED-20','Nancy Walker',75,'Bacterial Meningitis','WARN_O2'),
    ('BED-21','Paul Allen',58,'Post-Op Appendectomy','STABLE'),
    ('BED-22','Karen Young',65,'Controlled Diabetes','STABLE'),
    ('BED-23','Steven King',72,'Cardiac Monitoring','STABLE'),
    ('BED-24','Betty Wright',61,'Post-Cholecystectomy','STABLE'),
    ('BED-25','Ronald Scott',55,'Hypertension Management','STABLE'),
    ('BED-26','Margaret Green',78,'Hip Replacement Recovery','STABLE'),
    ('BED-27','Edward Baker',63,'Stable Angina','STABLE'),
    ('BED-28','Sandra Adams',49,'Asthma Management','STABLE'),
    ('BED-29','Daniel Nelson',70,'Urolithiasis Treatment','STABLE'),
    ('BED-30','Ruth Carter',82,'Atrial Fibrillation','STABLE'),
    ('BED-31','Gary Mitchell',54,'Lower GI Bleeding','STABLE'),
    ('BED-32','Carol Perez',67,'Cellulitis Treatment','STABLE'),
    ('BED-33','Larry Roberts',73,'Pulmonary Fibrosis','STABLE'),
    ('BED-34','Joyce Turner',56,'Post-Knee Replacement','STABLE'),
    ('BED-35','Jerry Phillips',60,'Hypertension Control','STABLE'),
    ('BED-36','Judith Campbell',77,'Dementia Complication','STABLE'),
    ('BED-37','Frank Parker',51,'Chest Pain Monitoring','STABLE'),
    ('BED-38','Evelyn Evans',84,'Dehydration Treatment','STABLE'),
    ('BED-39','Harold Edwards',69,'Post-CABG Recovery','STABLE'),
    ('BED-40','Diane Collins',58,'UTI Management','STABLE'),
    ('BED-41','Arthur Stewart',65,'Vertebral Fracture','STABLE'),
    ('BED-42','Virginia Sanchez',72,'Bowel Obstruction Recovery','STABLE'),
    ('BED-43','Keith Morris',47,'Trauma Recovery','STABLE'),
    ('BED-44','Mildred Rogers',80,'Pneumonia Recovering','STABLE'),
    ('BED-45','Andrew Reed',53,'Post-Appendectomy','STABLE'),
    ('BED-46','Deborah Cook',66,'DVT Treatment','STABLE'),
    ('BED-47','Raymond Bailey',74,'COPD Stable Phase','STABLE'),
    ('BED-48','Catherine Rivera',61,'Cardiac Arrhythmia','STABLE'),
    ('BED-49','Joseph Cooper',79,'Post-Hip Surgery','STABLE'),
    ('BED-50','Frances Richardson',55,'Metabolic Syndrome','STABLE'),
]

# Initialize ward
ward = ICUWard()
print("Initializing 50-bed ICU Ward...")
for bed_id, name, age, diagnosis, category in WARD_PATIENTS:
    ward.add_bed(bed_id, name, age, diagnosis)
    vitals = get_initial_vitals(category)
    bed    = ward.get_bed(bed_id)
    for hr, s, d, o2 in vitals:
        bed['tracker'].add_reading(hr, s, d, o2)
    last = vitals[-1]
    bed['sim_state'] = {'heart_rate':last[0],'bp_systolic':last[1],
                        'bp_diastolic':last[2],'oxygen':last[3]}

print(f"✓ All {len(ward.beds)} beds initialized")


# ════════════════════════════════════════════════════════════════
# FLASK APP + ROUTES
# ════════════════════════════════════════════════════════════════
app = Flask(__name__, template_folder=os.path.join(os.getcwd(), 'templates'))
CORS(app)


@app.route('/')
def index():
    return render_template('index.html')

@app.route('/api/patients')
def api_patients():
    return jsonify(ward.all_statuses())

@app.route('/api/status')
def api_status():
    bed_id = request.args.get('bed', 'BED-01')
    bed    = ward.get_bed(bed_id)
    if not bed:
        return jsonify({'status': 'NO_DATA'})
    if bed['info'].get('vacant', False):
        return jsonify({
            'status':       'VACANT',
            'bed_id':       bed_id,
            'prev_patient': bed['info'].get('prev_patient', '—'),
            'discharged_at':bed['info'].get('discharged_at', '—'),
            'stack_size':   0,
            'auto_pops':    bed['tracker'].stack.auto_pops
        })
    status = bed['tracker'].get_status()
    if status['status'] != 'NO_DATA':
        status['patient_name'] = bed['info']['name']
        status['diagnosis']    = bed['info']['diagnosis']
    return jsonify(status)

@app.route('/api/history')
def api_history():
    bed_id = request.args.get('bed', 'BED-01')
    n      = request.args.get('n', 20, type=int)
    bed    = ward.get_bed(bed_id)
    return jsonify(bed['tracker'].get_history(n) if bed else [])

@app.route('/api/stack-items')
def api_stack_items():
    bed_id = request.args.get('bed', 'BED-01')
    bed    = ward.get_bed(bed_id)
    return jsonify(bed['tracker'].stack.get_all() if bed else [])

@app.route('/api/add-reading', methods=['POST'])
def api_add_reading():
    d = request.get_json(force=True)
    if not d:
        return jsonify({'success':False,'error':'No data'}), 400
    bed = ward.get_bed(d.get('bed','BED-01'))
    if not bed:
        return jsonify({'success':False,'error':'Bed not found'}), 404
    if bed['info'].get('vacant', False):
        return jsonify({'success':False,'error':'Cannot add readings to a vacant bed'}), 400
    try:
        snap, auto_popped = bed['tracker'].add_reading(
            d['heart_rate'], d['bp_systolic'], d['bp_diastolic'], d['oxygen'])
        return jsonify({
            'success':    True,
            'stack_size': bed['tracker'].stack.size(),
            'auto_pop':   auto_popped is not None   # tells UI if auto-pop fired
        })
    except (KeyError, ValueError) as e:
        return jsonify({'success':False,'error':str(e)}), 400

@app.route('/api/simulate', methods=['POST'])
def api_simulate():
    d      = request.get_json(force=True) or {}
    result = ward.simulate_bed(d.get('bed','BED-01'))
    return jsonify({'success': result})

@app.route('/api/simulate-all', methods=['POST'])
def api_simulate_all():
    ward.simulate_all()
    return jsonify({'success': True})

@app.route('/api/simulate-crisis', methods=['POST'])
def api_simulate_crisis():
    d   = request.get_json(force=True) or {}
    bed = ward.get_bed(d.get('bed','BED-01'))
    if not bed:
        return jsonify({'success':False,'error':'Bed not found'})
    if bed['info'].get('vacant', False):
        return jsonify({'success':False,'error':'Bed is vacant — cannot simulate crisis'})
    latest  = bed['tracker'].stack.peek(0)
    base_hr = latest.heart_rate if latest else 80
    crisis  = [
        (max(40, int(base_hr * 0.88)), 116, 75, 94),
        (max(40, int(base_hr * 0.88 * 0.87)), 114, 74, 92),
        (max(40, int(base_hr * 0.88 * 0.87 * 0.86)), 112, 73, 90)
    ]
    for hr, s, dia, o2 in crisis:
        bed['tracker'].add_reading(hr, s, dia, o2)
    last = crisis[-1]
    bed['sim_state'] = {'heart_rate':last[0],'bp_systolic':last[1],
                        'bp_diastolic':last[2],'oxygen':last[3]}
    return jsonify({'success': True})

@app.route('/api/crisis-report')
def api_crisis_report():
    bed_id = request.args.get('bed','BED-01')
    bed    = ward.get_bed(bed_id)
    if not bed:
        return jsonify({'readings':[],'summary':'Bed not found'})
    if bed['info'].get('vacant', False):
        return jsonify({'readings':[],'summary':'Bed is vacant — no readings available.'})
    return jsonify(bed['tracker'].get_crisis_report())

@app.route('/api/discharge', methods=['POST'])
def api_discharge():
    """
    Patient Discharge — uses pop_all() to clear the entire stack.
    This demonstrates the POP operation at scale.
    """
    d      = request.get_json(force=True) or {}
    bed_id = d.get('bed', 'BED-01')
    bed    = ward.get_bed(bed_id)

    if not bed:
        return jsonify({'success':False,'error':'Bed not found'}), 404
    if bed['info'].get('vacant', False):
        return jsonify({'success':False,'error':'Bed is already vacant'}), 400

    patient_name    = bed['info']['name']
    readings_popped = bed['tracker'].discharge_patient()   # ← POP ALL

    # Mark bed as vacant, archive patient info
    bed['info']['vacant']         = True
    bed['info']['prev_patient']   = patient_name
    bed['info']['prev_diagnosis'] = bed['info']['diagnosis']
    bed['info']['name']           = 'VACANT'
    bed['info']['diagnosis']      = 'Bed Available for New Patient'
    bed['info']['discharged_at']  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # Reset simulator state for next patient
    bed['sim_state'] = {'heart_rate':80,'bp_systolic':120,'bp_diastolic':80,'oxygen':97}

    return jsonify({
        'success':        True,
        'patient':        patient_name,
        'bed_id':         bed_id,
        'readings_popped':readings_popped,
        'discharged_at':  bed['info']['discharged_at']
    })


def _gen_vitals(last):
    return {
        'heart_rate':   max(50, min(130, last['heart_rate']   + random.randint(-3,3))),
        'bp_systolic':  max(85, min(170, last['bp_systolic']  + random.randint(-2,2))),
        'bp_diastolic': max(50, min(110, last['bp_diastolic'] + random.randint(-1,1))),
        'oxygen':       max(82, min(100, last['oxygen']       + random.randint(-1,1)))
    }


print("=" * 58)
print("✓ Flask app ready — 3 POP operations implemented:")
print("  stack.pop()        — removes TOP (most recent) reading")
print("  stack.pop_oldest() — removes BOTTOM (oldest) reading")
print("  stack.pop_all()    — clears ALL readings (discharge)")
print("✓ Auto-POP at 100-item limit — tracked in auto_pops counter")
print("✓ /api/discharge route — triggers pop_all() for any bed")
print("✓ Vacant bed handling throughout all routes")
print("=" * 58)
print("Run Cell 4 to launch!")

Initializing 50-bed ICU Ward...
✓ All 50 beds initialized
✓ Flask app ready — 3 POP operations implemented:
  stack.pop()        — removes TOP (most recent) reading
  stack.pop_oldest() — removes BOTTOM (oldest) reading
  stack.pop_all()    — clears ALL readings (discharge)
✓ Auto-POP at 100-item limit — tracked in auto_pops counter
✓ /api/discharge route — triggers pop_all() for any bed
✓ Vacant bed handling throughout all routes
Run Cell 4 to launch!


In [7]:
# ================================================================
# CELL 4: Launch SENTINEL ICU
#
# NGROK TOKEN (get free at ngrok.com):
#   1. Go to ngrok.com → Sign Up Free
#   2. After login → left sidebar → "Your Authtoken"
#   3. Copy the token shown
#   4. Paste it below — ONLY change what's inside the quotes
#      Keep the quotes. Do not delete anything else.
# ================================================================

NGROK_TOKEN = "3DM9DU6sGryKXzMVXQ1bIR5VRMR_2RMbS8evvo1n2VjKgfyKC"

# ── DO NOT MODIFY ANYTHING BELOW THIS LINE ──────────────────────
import threading, time, subprocess
from pyngrok import ngrok

# Kill any old process using port 5000 (prevents "Address in use" error)
subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)
time.sleep(1)

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)
print("✓ Flask server started on port 5000")

# Try ngrok → fall back to Colab URL automatically
app_url = None

if "PASTE_YOUR" not in NGROK_TOKEN and len(NGROK_TOKEN) > 20:
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
        tunnel  = ngrok.connect(5000)
        app_url = tunnel.public_url
        print("✓ ngrok tunnel active")
    except Exception as e:
        print(f"⚠️  ngrok error: {e}")

if not app_url:
    try:
        from google.colab.output import eval_js
        app_url = eval_js("google.colab.kernel.proxyPort(5000)")
        print("✓ Using Colab URL (works perfectly!)")
    except:
        app_url = "http://localhost:5000"

print()
print("═" * 60)
print("🏥  SENTINEL ICU IS LIVE — 50-BED ICU WARD")
print("═" * 60)
print(f"🌐  OPEN THIS URL IN YOUR BROWSER:\n")
print(f"      {app_url}\n")
print("═" * 60)
print("📌  PRESENTATION GUIDE:")
print("  Tab 1 (Overview) → 50 beds, 10 CRITICAL shown immediately")
print("  Click any bed    → goes to Patient Monitor for that bed")
print("  Click Start All  → all 50 beds update live every 5 seconds")
print("  Tab 2 (Monitor)  → dropdown has all 50 patients")
print("                   → Simulate Crash → see 🚨 CRITICAL alert")
print("  Tab 3 (Reports)  → select patient → Crisis Report")
print("═" * 60)
print()
print("📋  PRESENTATION DAY INSTRUCTIONS:")
print("  1. Open this Colab notebook")
print("  2. Run Cell 3 → then Cell 4 (Cells 1 & 2 not needed again)")
print("  3. Copy the new URL from Cell 4 output")
print("  4. Keep this Colab tab open during entire presentation")
print("  5. URL works as long as Colab tab stays open")
print("  → Your existing ngrok token is fine — no need for new one")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✓ Flask server started on port 5000
✓ ngrok tunnel active

════════════════════════════════════════════════════════════
🏥  SENTINEL ICU IS LIVE — 50-BED ICU WARD
════════════════════════════════════════════════════════════
🌐  OPEN THIS URL IN YOUR BROWSER:

      https://untainted-angriness-poncho.ngrok-free.dev

════════════════════════════════════════════════════════════
📌  PRESENTATION GUIDE:
  Tab 1 (Overview) → 50 beds, 10 CRITICAL shown immediately
  Click any bed    → goes to Patient Monitor for that bed
  Click Start All  → all 50 beds update live every 5 seconds
  Tab 2 (Monitor)  → dropdown has all 50 patients
                   → Simulate Crash → see 🚨 CRITICAL alert
  Tab 3 (Reports)  → select patient → Crisis Report
════════════════════════════════════════════════════════════

📋  PRESENTATION DAY INSTRUCTIONS:
  1. Open this Colab notebook
  2. Run Cell 3 → then Cell 4 (Cells 1 & 2 not needed again)
  3. Copy the new URL from Cell 4 output
  4. Keep this Colab tab open dur